# Compare Retrievers (Colab Wrapper)

This notebook is a thin wrapper around `compare_retrievers.py`.

Run it after the custom retriever checkpoint has been created.
The intended order is:
1. train the custom retriever
2. run retriever-only comparison
3. optionally run fixed-generator evaluation later

This notebook does not duplicate benchmark logic. It mounts Drive, enters the repo root, installs dependencies, checks required paths, discovers valid benchmark options from the synced `WildGraphBench/` tree, and launches the existing script in a fresh Python process.


In [1]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
import os
from pathlib import Path

# Update this to your Drive-synced repo path before running.
REPO_ROOT = Path("/content/drive/MyDrive/UVM/Deep Learning/Final Project/repo-mirror/deep-learning-final-project")
if not REPO_ROOT.exists():
    raise FileNotFoundError(f"Repo root not found: {REPO_ROOT}")

os.chdir(REPO_ROOT)
print(f"Working directory: {Path.cwd()}")


Working directory: /content/drive/MyDrive/UVM/Deep Learning/Final Project/repo-mirror/deep-learning-final-project


In [3]:
%pip install -r requirements.txt
%pip install sentence-transformers transformers datasets accelerate peft trl scikit-learn matplotlib beautifulsoup4


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [4]:
from pathlib import Path

required_paths = [
    Path("compare_retrievers.py"),
    Path("WildGraphBench"),
    Path("all_embeddings"),
    Path("custom_all_embeddings"),
]

missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required files for retriever comparison:\n- " + "\n- ".join(missing)
    )

corpus_root = Path("WildGraphBench/corpus")
qa_root = Path("WildGraphBench/QA")
available_pairs = []

for domain_dir in sorted(corpus_root.iterdir()):
    if not domain_dir.is_dir():
        continue
    qa_path = qa_root / domain_dir.name / "questions.jsonl"
    if not qa_path.exists():
        continue
    for topic_dir in sorted(domain_dir.iterdir()):
        if not topic_dir.is_dir():
            continue
        if (topic_dir / "reference_pages").exists():
            available_pairs.append((domain_dir.name, topic_dir.name))

if not available_pairs:
    raise FileNotFoundError("No valid WildGraphBench domain/topic pairs were found in the synced repo.")

print("Available WildGraphBench domain/topic pairs:")
for domain, topic in available_pairs:
    print(f"- {domain} / {topic}")


Available WildGraphBench domain/topic pairs:
- culture / Marvel Cinematic Universe
- geography / United States
- health / COVID-19 pandemic
- history / World War II
- human_activities / 2022 FIFA World Cup
- mathematics / Prime number
- nature / 2012 Pacific typhoon season
- people / Donald Trump
- philosophy / Authoritarian socialism
- religion / Persecution of Muslims
- society / Human
- technology / Steam(service)


In [5]:
# Default to the first valid on-disk pair discovered above.
# Replace these values with another discovered pair if you want a different benchmark slice.
DOMAIN, TOPIC = available_pairs[0]

WILDBENCH_ROOT = "WildGraphBench"
WILDGRAPH_EMBEDDER_PATH = "./all_embeddings"
CUSTOM_EMBEDDER_PATH = "./custom_all_embeddings"

INCLUDE_BASE = False
K = 3
LIMIT = 50
CHUNK_SIZE = 300
WORD_COVERAGE_THRESHOLD = 0.5
DEVICE = "cuda"

print(f"Selected domain/topic: {DOMAIN} / {TOPIC}")


Selected domain/topic: culture / Marvel Cinematic Universe


In [6]:
from pathlib import Path

qa_path = Path(WILDBENCH_ROOT) / "QA" / DOMAIN / "questions.jsonl"
reference_dir = Path(WILDBENCH_ROOT) / "corpus" / DOMAIN / TOPIC / "reference_pages"

if not qa_path.exists():
    raise FileNotFoundError(f"QA file not found: {qa_path}")
if not reference_dir.exists():
    raise FileNotFoundError(f"Reference pages directory not found: {reference_dir}")

print(f"QA file: {qa_path}")
print(f"Reference pages directory: {reference_dir}")


QA file: WildGraphBench/QA/culture/questions.jsonl
Reference pages directory: WildGraphBench/corpus/culture/Marvel Cinematic Universe/reference_pages


In [7]:
import shlex

compare_cmd = [
    "python",
    "compare_retrievers.py",
    "--repo_path", WILDBENCH_ROOT,
    "--wildgraph_embedder_path", WILDGRAPH_EMBEDDER_PATH,
    "--custom_embedder_path", CUSTOM_EMBEDDER_PATH,
    "--domain", DOMAIN,
    "--topic", TOPIC,
    "--k", str(K),
    "--limit", str(LIMIT),
    "--chunk_size", str(CHUNK_SIZE),
    "--word_coverage_threshold", str(WORD_COVERAGE_THRESHOLD),
    "--device", DEVICE,
]

if INCLUDE_BASE:
    compare_cmd.append("--include_base")

compare_cmd_str = " ".join(shlex.quote(part) for part in compare_cmd)
print(compare_cmd_str)
!{compare_cmd_str}


python compare_retrievers.py --repo_path WildGraphBench --wildgraph_embedder_path ./all_embeddings --custom_embedder_path ./custom_all_embeddings --domain culture --topic 'Marvel Cinematic Universe' --k 3 --limit 50 --chunk_size 300 --word_coverage_threshold 0.5 --device cuda
Loading weights: 100% 103/103 [00:08<00:00, 11.49it/s, Materializing param=pooler.dense.weight]
Loading weights: 100% 103/103 [00:04<00:00, 20.62it/s, Materializing param=pooler.dense.weight]
Evaluating on 50 QA examples from domain='culture', topic='Marvel Cinematic Universe'
Reference chunks: 3120

=== Retriever Comparison ===
wildgraph  Recall@3: 0.520
custom     Recall@3: 0.460


## Optional Fixed-Generator Evaluation

Only run this section after you have completed the retriever-only comparison and decided the fixed-generator check is worth the additional cost.


In [8]:
import shlex

RUN_FIXED_GENERATOR = False
LORA_CHECKPOINT = "./tinyllama-lora-mcu"
BASE_MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
N_GEN = 20

if RUN_FIXED_GENERATOR:
    optional_cmd = [
        "python",
        "compare_retrievers.py",
        "--repo_path", WILDBENCH_ROOT,
        "--wildgraph_embedder_path", WILDGRAPH_EMBEDDER_PATH,
        "--custom_embedder_path", CUSTOM_EMBEDDER_PATH,
        "--domain", DOMAIN,
        "--topic", TOPIC,
        "--k", str(K),
        "--limit", str(LIMIT),
        "--chunk_size", str(CHUNK_SIZE),
        "--word_coverage_threshold", str(WORD_COVERAGE_THRESHOLD),
        "--lora_checkpoint", LORA_CHECKPOINT,
        "--base_model_name", BASE_MODEL_NAME,
        "--n_gen", str(N_GEN),
        "--device", DEVICE,
    ]
    optional_cmd_str = " ".join(shlex.quote(part) for part in optional_cmd)
    print(optional_cmd_str)
    !{optional_cmd_str}
else:
    print("Set RUN_FIXED_GENERATOR = True if you want the optional fixed-generator evaluation.")


Set RUN_FIXED_GENERATOR = True if you want the optional fixed-generator evaluation.
